# Assignment 2 — Attention + Decoding Strategies

## Task

Extend Task 1's basic English → French LSTM encoder-decoder with an attention mechanism and compare inference-time decoding strategies.

### Requirements
1. Add an attention mechanism to Task 1's encoder-decoder.
2. Re-train on the **same dataset and same train/validation/test splits**.
3. Evaluate with corpus BLEU on the held-out test set.
4. Compare:
   - Task 1: No attention
   - Task 2: Bahdanau attention
   - Greedy decoding
   - Beam search
   - Top-k sampling
5. Analyze examples, sentence length, rare words, repetitions, and other failure modes.

### Architecture

**Task 1**

`English → LSTM Encoder → fixed final state → LSTM Decoder → French`

**Task 2**

`English → LSTM Encoder → all encoder states → Bahdanau Attention ↔ Decoder → French`

The central change is that the decoder can dynamically access all encoder states instead of relying on one fixed context representation.

### Experimental principle

This is a direct extension of Task 1. The dataset, split, tokenization philosophy, vocabulary construction, and main model dimensions are kept the same wherever possible so that attention is the principal architectural change.

In [ ]:
# ============================================================
# 1. IMPORTS + REPRODUCIBILITY
# ============================================================

import os
import re
import random
import time
import zipfile
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from tqdm.auto import tqdm

from nltk.translate.bleu_score import (
    corpus_bleu,
    sentence_bleu,
    SmoothingFunction
)

import nltk
nltk.download("punkt", quiet=True)

# Reproducibility: keep all random sources fixed.
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)

torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 2. Dataset

The supplied files are the same files used for Task 1:

- `en_fr_train(1).zip`
- `en_fr_val(1).csv`
- `en_fr_test(1).csv`

The ZIP contains the training CSV. The vocabulary is built from the training split only to avoid validation/test leakage.

In [ ]:
# ============================================================
# 2. LOAD THE SAME TASK-1 DATASET
# ============================================================

TRAIN_ZIP = "/mnt/data/en_fr_train(1).zip"
VAL_PATH = "/mnt/data/en_fr_val(1).csv"
TEST_PATH = "/mnt/data/en_fr_test(1).csv"

EXTRACT_DIR = "/mnt/data/en_fr_train_extracted"
os.makedirs(EXTRACT_DIR, exist_ok=True)

with zipfile.ZipFile(TRAIN_ZIP, "r") as z:
    z.extractall(EXTRACT_DIR)

# Find the CSV inside the training ZIP automatically.
train_csv = None
for root, dirs, files in os.walk(EXTRACT_DIR):
    for file in files:
        if file.lower().endswith(".csv"):
            train_csv = os.path.join(root, file)
            break
    if train_csv:
        break

if train_csv is None:
    raise FileNotFoundError("No CSV found inside the training ZIP.")

train_df = pd.read_csv(train_csv)
val_df = pd.read_csv(VAL_PATH)
test_df = pd.read_csv(TEST_PATH)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)
print("Columns:", train_df.columns.tolist())

In [ ]:
# ============================================================
# 3. STANDARDIZE + CLEAN
# ============================================================

# The supplied columns are:
#   en = source English
#   fr = target French
#
# Generic names make the rest of the implementation clearer.

train_df = train_df.rename(columns={"en": "src", "fr": "trg"})
val_df = val_df.rename(columns={"en": "src", "fr": "trg"})
test_df = test_df.rename(columns={"en": "src", "fr": "trg"})

train_df = train_df.dropna(subset=["src", "trg"]).reset_index(drop=True)
val_df = val_df.dropna(subset=["src", "trg"]).reset_index(drop=True)
test_df = test_df.dropna(subset=["src", "trg"]).reset_index(drop=True)

display(train_df.head())

In [ ]:
# ============================================================
# 4. TOKENIZATION
# ============================================================

def tokenize(text):
    """
    Simple word-level tokenizer matching the Task-1 approach:
    lowercase, separate common punctuation, normalize spaces.
    """
    text = str(text).lower()
    text = re.sub(r"([?.!,;:()\[\]{}\"'])", r" \1 ", text)
    text = re.sub(r"\s+", " ", text).strip()
    return text.split()

print(train_df.iloc[0]["src"])
print(tokenize(train_df.iloc[0]["src"]))

In [ ]:
# ============================================================
# 5. VOCABULARY
# ============================================================

PAD_TOKEN = "<pad>"
UNK_TOKEN = "<unk>"
SOS_TOKEN = "<sos>"
EOS_TOKEN = "<eos>"


class Vocabulary:
    def __init__(self, min_freq=2):
        self.itos = [PAD_TOKEN, UNK_TOKEN, SOS_TOKEN, EOS_TOKEN]
        self.stoi = {
            PAD_TOKEN: 0,
            UNK_TOKEN: 1,
            SOS_TOKEN: 2,
            EOS_TOKEN: 3
        }
        self.min_freq = min_freq

    def build(self, sentences):
        # IMPORTANT: call this only on training sentences.
        counter = Counter()
        for sentence in sentences:
            counter.update(tokenize(sentence))

        for token, freq in counter.items():
            if freq >= self.min_freq and token not in self.stoi:
                self.stoi[token] = len(self.itos)
                self.itos.append(token)

    def numericalize(self, sentence):
        # Every sequence has explicit start/end markers.
        ids = [
            self.stoi.get(token, self.stoi[UNK_TOKEN])
            for token in tokenize(sentence)
        ]
        return [self.stoi[SOS_TOKEN]] + ids + [self.stoi[EOS_TOKEN]]

    def decode(self, indices):
        tokens = []
        for idx in indices:
            token = self.itos[int(idx)]
            if token == EOS_TOKEN:
                break
            if token not in {PAD_TOKEN, SOS_TOKEN}:
                tokens.append(token)
        return tokens

    def __len__(self):
        return len(self.itos)


src_vocab = Vocabulary(min_freq=2)
trg_vocab = Vocabulary(min_freq=2)

# No validation/test text is used to build the vocabulary.
src_vocab.build(train_df["src"])
trg_vocab.build(train_df["trg"])

SRC_PAD_IDX = src_vocab.stoi[PAD_TOKEN]
TRG_PAD_IDX = trg_vocab.stoi[PAD_TOKEN]

print("Source vocabulary:", len(src_vocab))
print("Target vocabulary:", len(trg_vocab))

In [ ]:
# ============================================================
# 6. DATASET + DATALOADERS
# ============================================================

# Execution-oriented limits.
# These reduce recurrent time while preserving the same model architecture.
MAX_SRC_LEN = 60
MAX_TRG_LEN = 60

class TranslationDataset(Dataset):
    def __init__(self, dataframe, src_vocab, trg_vocab,
                 max_src_len=MAX_SRC_LEN, max_trg_len=MAX_TRG_LEN):
        self.df = dataframe.reset_index(drop=True)
        self.src_vocab = src_vocab
        self.trg_vocab = trg_vocab
        self.max_src_len = max_src_len
        self.max_trg_len = max_trg_len

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        src_tokens = tokenize(self.df.iloc[idx]["src"])[:self.max_src_len]
        trg_tokens = tokenize(self.df.iloc[idx]["trg"])[:self.max_trg_len]

        src = self.src_vocab.numericalize(
            " ".join(src_tokens)
        )
        trg = self.trg_vocab.numericalize(
            " ".join(trg_tokens)
        )

        return (
            torch.tensor(src, dtype=torch.long),
            torch.tensor(trg, dtype=torch.long)
        )


def collate_fn(batch):
    # Shape after padding: [sequence_length, batch_size]
    src = [x[0] for x in batch]
    trg = [x[1] for x in batch]

    src = pad_sequence(src, padding_value=SRC_PAD_IDX)
    trg = pad_sequence(trg, padding_value=TRG_PAD_IDX)

    return src, trg


train_dataset = TranslationDataset(
    train_df, src_vocab, trg_vocab
)
val_dataset = TranslationDataset(
    val_df, src_vocab, trg_vocab
)
test_dataset = TranslationDataset(
    test_df, src_vocab, trg_vocab
)

BATCH_SIZE = 64

# pin_memory helps host-to-GPU transfer when CUDA is available.
PIN_MEMORY = torch.cuda.is_available()

train_loader = DataLoader(
    train_dataset,
    batch_size=BATCH_SIZE,
    shuffle=True,
    collate_fn=collate_fn,
    pin_memory=PIN_MEMORY
)
val_loader = DataLoader(
    val_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=PIN_MEMORY
)
test_loader = DataLoader(
    test_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    collate_fn=collate_fn,
    pin_memory=PIN_MEMORY
)

print("Max source tokens:", MAX_SRC_LEN)
print("Max target tokens:", MAX_TRG_LEN)
print("Batch size:", BATCH_SIZE)


# 7. Bahdanau Attention

For source encoder states \(h_1,\ldots,h_{T_x}\) and previous decoder state \(s_{t-1}\):

\[
e_{t,i}=v_a^T\tanh(W_hh_i+W_ss_{t-1})
\]

Normalize the alignment scores:

\[
\alpha_{t,i}=
\frac{\exp(e_{t,i})}
{\sum_j\exp(e_{t,j})}
\]

Then compute the dynamic context:

\[
c_t=\sum_i\alpha_{t,i}h_i
\]

The key difference from Task 1 is that \(c_t\) changes for every target timestep.

In [ ]:
# ============================================================
# 7. HYPERPARAMETERS — EXECUTION-OPTIMIZED CONFIGURATION
# ============================================================
#
# The architecture remains the same, but the model is smaller and
# training/sequence limits are reduced for substantially faster execution.
#
# Fair-comparison note:
# If strict capacity matching against Task 1 is required, use the same
# reduced dimensions for Task 1 as well.

INPUT_DIM = len(src_vocab)
OUTPUT_DIM = len(trg_vocab)

ENC_EMB_DIM = 128
DEC_EMB_DIM = 128
HID_DIM = 256

N_LAYERS = 1
ENC_DROPOUT = 0.2
DEC_DROPOUT = 0.2

LEARNING_RATE = 0.001

N_EPOCHS = 10
PATIENCE = 2
CLIP = 1.0

MAX_DECODE_LEN = 60

BEAM_WIDTH = 5
TOP_K = 5
TOP_K_TEMPERATURE = 0.8

print("Embedding dimension:", ENC_EMB_DIM)
print("Hidden dimension:", HID_DIM)
print("Maximum epochs:", N_EPOCHS)
print("Early-stopping patience:", PATIENCE)
print("Maximum source/target tokens:", MAX_SRC_LEN)
print("Maximum decode length:", MAX_DECODE_LEN)


### Execution note

This notebook uses a smaller 128-dimensional embedding / 256-dimensional hidden-state configuration, 10 maximum epochs with patience 2, 60-token sequence limits, `tqdm` progress bars, and a 60-token decoding limit. These changes reduce runtime without removing Bahdanau attention or any required decoding strategy.

For a strict attention-vs-no-attention capacity comparison, report that this optimized configuration is smaller than the original 256/512 setup, or rerun Task 1 with the same reduced dimensions.


In [ ]:
# ============================================================
# 8. LSTM ENCODER
# ============================================================

class Encoder(nn.Module):
    def __init__(
        self, input_dim, embedding_dim, hidden_dim,
        n_layers=1, dropout=0.0
    ):
        super().__init__()

        self.embedding = nn.Embedding(
            input_dim, embedding_dim,
            padding_idx=SRC_PAD_IDX
        )

        self.rnn = nn.LSTM(
            embedding_dim,
            hidden_dim,
            num_layers=n_layers,
            dropout=dropout if n_layers > 1 else 0
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, src):
        # Unlike Task 1, ALL encoder outputs are retained.
        # Attention needs h_1, h_2, ..., h_T rather than only h_T.
        embedded = self.dropout(self.embedding(src))
        outputs, (hidden, cell) = self.rnn(embedded)
        return outputs, hidden, cell

In [ ]:
# ============================================================
# 9. BAHDANAU ADDITIVE ATTENTION
# ============================================================

class BahdanauAttention(nn.Module):
    def __init__(
        self,
        encoder_hidden_dim,
        decoder_hidden_dim,
        attention_dim
    ):
        super().__init__()

        # Project encoder and decoder states into the same
        # attention space before combining them.
        self.encoder_projection = nn.Linear(
            encoder_hidden_dim, attention_dim, bias=False
        )
        self.decoder_projection = nn.Linear(
            decoder_hidden_dim, attention_dim, bias=False
        )

        # Convert the combined representation into a scalar
        # alignment score for each source position.
        self.energy = nn.Linear(
            attention_dim, 1, bias=False
        )

    def forward(
        self,
        decoder_hidden,
        encoder_outputs,
        mask
    ):
        """
        decoder_hidden:
            [batch, decoder_hidden_dim]

        encoder_outputs:
            [src_len, batch, encoder_hidden_dim]

        mask:
            [batch, src_len]

        returns:
            attention weights [batch, src_len]
        """

        # [batch, src_len, encoder_hidden_dim]
        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        # [batch, src_len, attention_dim]
        encoder_features = self.encoder_projection(
            encoder_outputs
        )

        # [batch, 1, attention_dim]
        decoder_features = self.decoder_projection(
            decoder_hidden
        ).unsqueeze(1)

        # Bahdanau additive score:
        # e_ti = v^T tanh(W_h h_i + W_s s_{t-1})
        energy = torch.tanh(
            encoder_features + decoder_features
        )

        # [batch, src_len, 1] -> [batch, src_len]
        energy = self.energy(energy).squeeze(2)

        # Padding tokens must never receive attention.
        # A very negative value becomes ~0 after softmax.
        energy = energy.masked_fill(mask == 0, -1e10)

        # Convert scores into a probability distribution
        # across source positions.
        return torch.softmax(energy, dim=1)

In [ ]:
# ============================================================
# 10. ATTENTION DECODER
# ============================================================

class AttentionDecoder(nn.Module):
    def __init__(
        self,
        output_dim,
        embedding_dim,
        encoder_hidden_dim,
        decoder_hidden_dim,
        attention,
        n_layers=1,
        dropout=0.0
    ):
        super().__init__()

        self.output_dim = output_dim
        self.attention = attention

        self.embedding = nn.Embedding(
            output_dim, embedding_dim,
            padding_idx=TRG_PAD_IDX
        )

        # At each step the LSTM receives:
        #   previous target embedding + attention context.
        self.rnn = nn.LSTM(
            embedding_dim + encoder_hidden_dim,
            decoder_hidden_dim,
            num_layers=n_layers,
            dropout=dropout if n_layers > 1 else 0
        )

        # Prediction uses decoder output, context and embedding.
        self.fc_out = nn.Linear(
            embedding_dim + encoder_hidden_dim + decoder_hidden_dim,
            output_dim
        )

        self.dropout = nn.Dropout(dropout)

    def forward(
        self,
        input_token,
        hidden,
        cell,
        encoder_outputs,
        mask
    ):
        # ----------------------------------------------------
        # 1. Embed previous target token
        # ----------------------------------------------------
        input_token = input_token.unsqueeze(0)
        embedded = self.dropout(
            self.embedding(input_token)
        )

        # ----------------------------------------------------
        # 2. Calculate attention weights
        # ----------------------------------------------------
        # The current decoder state tells attention what
        # source information is useful right now.
        decoder_hidden = hidden[-1]

        attention_weights = self.attention(
            decoder_hidden,
            encoder_outputs,
            mask
        )

        # ----------------------------------------------------
        # 3. Build context vector
        # ----------------------------------------------------
        # c_t = sum_i alpha_ti h_i
        weights = attention_weights.unsqueeze(1)

        encoder_batch = encoder_outputs.permute(1, 0, 2)

        context = torch.bmm(weights, encoder_batch)
        context = context.permute(1, 0, 2)

        # ----------------------------------------------------
        # 4. Feed embedding + context to decoder LSTM
        # ----------------------------------------------------
        rnn_input = torch.cat(
            (embedded, context),
            dim=2
        )

        output, (hidden, cell) = self.rnn(
            rnn_input,
            (hidden, cell)
        )

        # ----------------------------------------------------
        # 5. Predict next target token
        # ----------------------------------------------------
        output = output.squeeze(0)
        embedded = embedded.squeeze(0)
        context = context.squeeze(0)

        prediction = self.fc_out(
            torch.cat(
                (output, context, embedded),
                dim=1
            )
        )

        return (
            prediction,
            hidden,
            cell,
            attention_weights
        )

In [ ]:
# ============================================================
# 11. SEQ2SEQ WRAPPER
# ============================================================

class AttentionSeq2Seq(nn.Module):
    def __init__(self, encoder, decoder, src_pad_idx, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.src_pad_idx = src_pad_idx
        self.device = device

    def create_mask(self, src):
        # True = real source token; False = padding.
        return (src != self.src_pad_idx).permute(1, 0)

    def forward(
        self,
        src,
        trg,
        teacher_forcing_ratio=0.5
    ):
        batch_size = trg.shape[1]
        trg_len = trg.shape[0]
        trg_vocab_size = self.decoder.output_dim

        outputs = torch.zeros(
            trg_len,
            batch_size,
            trg_vocab_size,
            device=self.device
        )

        # Encoder produces all source states.
        encoder_outputs, hidden, cell = self.encoder(src)

        mask = self.create_mask(src)

        # First decoder input is <sos>.
        input_token = trg[0]

        for t in range(1, trg_len):

            output, hidden, cell, _ = self.decoder(
                input_token,
                hidden,
                cell,
                encoder_outputs,
                mask
            )

            outputs[t] = output

            # Teacher forcing exposes the true previous target
            # token during part of training.
            teacher_force = (
                random.random() < teacher_forcing_ratio
            )

            predicted = output.argmax(dim=1)

            input_token = (
                trg[t] if teacher_force else predicted
            )

        return outputs

In [ ]:
# ============================================================
# 12. CREATE MODEL
# ============================================================

attention = BahdanauAttention(
    encoder_hidden_dim=HID_DIM,
    decoder_hidden_dim=HID_DIM,
    attention_dim=HID_DIM
)

encoder = Encoder(
    INPUT_DIM, ENC_EMB_DIM, HID_DIM,
    N_LAYERS, ENC_DROPOUT
)

decoder = AttentionDecoder(
    OUTPUT_DIM, DEC_EMB_DIM,
    HID_DIM, HID_DIM,
    attention, N_LAYERS, DEC_DROPOUT
)

attention_model = AttentionSeq2Seq(
    encoder, decoder, SRC_PAD_IDX, device
).to(device)

def count_parameters(model):
    return sum(
        p.numel() for p in model.parameters()
        if p.requires_grad
    )

print("Trainable parameters:", count_parameters(attention_model))

## 13. Training

The loss is token-level cross entropy. `<pad>` is ignored.

The best checkpoint is selected using validation loss. The test set is not used for model selection.

In [ ]:
# ============================================================
# 13. TRAINING + VALIDATION
# ============================================================

criterion = nn.CrossEntropyLoss(
    ignore_index=TRG_PAD_IDX
)

optimizer = optim.Adam(
    attention_model.parameters(),
    lr=LEARNING_RATE
)


def train(model, iterator, optimizer, criterion, clip):
    model.train()
    epoch_loss = 0.0

    progress_bar = tqdm(
        iterator,
        desc="Training",
        leave=False
    )

    for src, trg in progress_bar:
        src = src.to(device, non_blocking=True)
        trg = trg.to(device, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        output = model(
            src,
            trg,
            teacher_forcing_ratio=0.5
        )

        output_dim = output.shape[-1]

        # Remove the <sos> timestep before calculating loss.
        output = output[1:].reshape(-1, output_dim)
        trg_loss = trg[1:].reshape(-1)

        loss = criterion(output, trg_loss)
        loss.backward()

        torch.nn.utils.clip_grad_norm_(
            model.parameters(), clip
        )

        optimizer.step()
        epoch_loss += loss.item()

        progress_bar.set_postfix(
            loss=f"{loss.item():.4f}"
        )

    return epoch_loss / len(iterator)


def evaluate(model, iterator, criterion):
    model.eval()
    epoch_loss = 0.0

    for src, trg in iterator:
        src = src.to(device, non_blocking=True)
        trg = trg.to(device, non_blocking=True)

        # No teacher forcing during validation.
        output = model(
            src,
            trg,
            teacher_forcing_ratio=0
        )

        output_dim = output.shape[-1]
        output = output[1:].reshape(-1, output_dim)
        trg_loss = trg[1:].reshape(-1)

        loss = criterion(output, trg_loss)
        epoch_loss += loss.item()

    return epoch_loss / len(iterator)


best_valid_loss = float("inf")
patience_counter = 0
train_losses = []
valid_losses = []

MODEL_PATH = "experiment2_attention_best.pt"

for epoch in range(N_EPOCHS):
    start = time.time()

    train_loss = train(
        attention_model,
        train_loader,
        optimizer,
        criterion,
        CLIP
    )

    valid_loss = evaluate(
        attention_model,
        val_loader,
        criterion
    )

    train_losses.append(train_loss)
    valid_losses.append(valid_loss)

    improved = valid_loss < best_valid_loss

    if improved:
        best_valid_loss = valid_loss
        patience_counter = 0
        torch.save(
            attention_model.state_dict(),
            MODEL_PATH
        )
    else:
        patience_counter += 1

    print(
        f"Epoch {epoch+1:02d}/{N_EPOCHS} | "
        f"Train {train_loss:.4f} | "
        f"Val {valid_loss:.4f} | "
        f"Time {time.time()-start:.1f}s"
    )

    if improved:
        print("  ✓ Best validation checkpoint saved.")
    else:
        print(
            f"  No improvement "
            f"({patience_counter}/{PATIENCE})"
        )

    if patience_counter >= PATIENCE:
        print("Early stopping.")
        break

# Always evaluate the best validation checkpoint, not the last epoch.
attention_model.load_state_dict(
    torch.load(MODEL_PATH, map_location=device)
)

print("Best validation loss:", best_valid_loss)
print("Best checkpoint:", MODEL_PATH)


In [ ]:
# ============================================================
# 14. TRAINING CURVE
# ============================================================

plt.figure(figsize=(10, 5))
plt.plot(train_losses, label="Training loss")
plt.plot(valid_losses, label="Validation loss")
plt.xlabel("Epoch")
plt.ylabel("Loss")
plt.title("Experiment 2 — Bahdanau Attention Seq2Seq")
plt.legend()
plt.grid(alpha=0.3)
plt.show()

# 15. Decoding Strategies

### Greedy decoding

Select the highest-probability next token:

\[
y_t=\arg\max_y P(y_t\mid y_{<t},x)
\]

Fast, deterministic, but an early local choice cannot be revised.

### Beam search

Maintain several candidate sequences simultaneously. Here we use beam width 5 and length-normalized cumulative log probability.

### Top-k sampling

Keep the top \(k\) next-token candidates and sample from them. This is stochastic and is primarily included for qualitative comparison.

In [ ]:
# ============================================================
# 15. SOURCE/PREDICTION HELPERS
# ============================================================

def prepare_source_sentence(sentence):
    ids = src_vocab.numericalize(sentence)
    return torch.tensor(
        ids,
        dtype=torch.long,
        device=device
    ).unsqueeze(1)


def ids_to_sentence(ids):
    return " ".join(trg_vocab.decode(ids))

In [ ]:
# ============================================================
# 16. GREEDY DECODING
# ============================================================

def greedy_decode(model, src, max_len=MAX_DECODE_LEN):
    """
    Keep only the highest-probability token at every step.
    Also return attention weights so we can visualize them.
    """
    model.eval()

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src)
        mask = model.create_mask(src)

        input_token = torch.tensor(
            [trg_vocab.stoi[SOS_TOKEN]],
            dtype=torch.long,
            device=device
        )

        generated = []
        attention_history = []

        for _ in range(max_len):
            output, hidden, cell, attn = model.decoder(
                input_token,
                hidden,
                cell,
                encoder_outputs,
                mask
            )

            predicted = output.argmax(dim=1)
            token_id = predicted.item()

            attention_history.append(
                attn.squeeze(0).cpu().numpy()
            )

            if token_id == trg_vocab.stoi[EOS_TOKEN]:
                break

            generated.append(token_id)
            input_token = predicted

    return generated, attention_history

In [ ]:
# ============================================================
# 17. BEAM SEARCH
# ============================================================

def beam_search_decode(
    model,
    src,
    beam_width=5,
    max_len=MAX_DECODE_LEN,
    length_penalty=0.7
):
    """
    Beam search keeps multiple partial translations.

    Each hypothesis stores:
      sequence
      cumulative log probability
      hidden state
      cell state

    Length normalization reduces the tendency to prefer very
    short sequences merely because they contain fewer negative
    log-probability terms.
    """
    model.eval()

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src)
        mask = model.create_mask(src)

        sos_id = trg_vocab.stoi[SOS_TOKEN]
        eos_id = trg_vocab.stoi[EOS_TOKEN]

        beams = [([sos_id], 0.0, hidden, cell)]

        for _ in range(max_len):
            candidates = []

            for sequence, score, h, c in beams:
                if sequence[-1] == eos_id:
                    candidates.append((sequence, score, h, c))
                    continue

                input_token = torch.tensor(
                    [sequence[-1]],
                    dtype=torch.long,
                    device=device
                )

                output, new_h, new_c, _ = model.decoder(
                    input_token,
                    h, c,
                    encoder_outputs,
                    mask
                )

                log_probs = torch.log_softmax(output, dim=1)

                top_values, top_indices = torch.topk(
                    log_probs,
                    beam_width,
                    dim=1
                )

                for k in range(beam_width):
                    token_id = top_indices[0, k].item()
                    token_score = top_values[0, k].item()

                    candidates.append((
                        sequence + [token_id],
                        score + token_score,
                        new_h,
                        new_c
                    ))

            candidates.sort(
                key=lambda x:
                    x[1] / (len(x[0]) ** length_penalty),
                reverse=True
            )

            beams = candidates[:beam_width]

            if all(
                beam[0][-1] == eos_id
                for beam in beams
            ):
                break

        best = max(
            beams,
            key=lambda x:
                x[1] / (len(x[0]) ** length_penalty)
        )[0]

    best = best[1:]

    if eos_id in best:
        best = best[:best.index(eos_id)]

    return best

In [ ]:
# ============================================================
# 18. TOP-K SAMPLING
# ============================================================

def top_k_sampling_decode(
    model,
    src,
    k=5,
    temperature=0.8,
    max_len=MAX_DECODE_LEN
):
    """
    Stochastic decoder:
      1. keep top-k tokens
      2. apply temperature
      3. sample one token

    Unlike greedy/beam decoding, repeated runs can differ.
    """
    model.eval()

    with torch.no_grad():
        encoder_outputs, hidden, cell = model.encoder(src)
        mask = model.create_mask(src)

        input_token = torch.tensor(
            [trg_vocab.stoi[SOS_TOKEN]],
            dtype=torch.long,
            device=device
        )

        generated = []

        for _ in range(max_len):
            output, hidden, cell, _ = model.decoder(
                input_token,
                hidden,
                cell,
                encoder_outputs,
                mask
            )

            logits = output / temperature

            top_values, top_indices = torch.topk(
                logits, k, dim=1
            )

            probabilities = torch.softmax(
                top_values, dim=1
            )

            sampled_position = torch.multinomial(
                probabilities, 1
            )

            predicted = top_indices[
                0, sampled_position
            ]

            token_id = predicted.item()

            if token_id == trg_vocab.stoi[EOS_TOKEN]:
                break

            generated.append(token_id)
            input_token = predicted

    return generated

In [ ]:
# ============================================================
# 19. USER-FACING DECODERS
# ============================================================

def translate_greedy(sentence):
    src = prepare_source_sentence(sentence)
    ids, _ = greedy_decode(
        attention_model,
        src,
        max_len=MAX_DECODE_LEN
    )
    return ids_to_sentence(ids)


def translate_beam(sentence, beam_width=BEAM_WIDTH):
    src = prepare_source_sentence(sentence)
    ids = beam_search_decode(
        attention_model,
        src,
        beam_width=beam_width,
        max_len=MAX_DECODE_LEN
    )
    return ids_to_sentence(ids)


def translate_sampling(
    sentence,
    k=TOP_K,
    temperature=TOP_K_TEMPERATURE
):
    src = prepare_source_sentence(sentence)
    ids = top_k_sampling_decode(
        attention_model,
        src,
        k=k,
        temperature=temperature,
        max_len=MAX_DECODE_LEN
    )
    return ids_to_sentence(ids)


In [ ]:
# ============================================================
# 20. QUALITATIVE DECODING COMPARISON
# ============================================================

examples = [
    "This is a very important problem.",
    "The world is changing rapidly.",
    "I think this is an interesting idea.",
    "We need to understand the problem.",
    "The man is standing near the door."
]

for sentence in examples:
    print("=" * 80)
    print("SOURCE:", sentence)
    print("GREEDY:", translate_greedy(sentence))
    print("BEAM-5:", translate_beam(sentence, BEAM_WIDTH))
    print(
        "TOP-K:",
        translate_sampling(
            sentence,
            k=TOP_K,
            temperature=TOP_K_TEMPERATURE
        )
    )


# 21. Test Evaluation

Corpus BLEU is computed on exactly the same held-out test set used by Task 1.

Greedy and beam search are evaluated quantitatively. Top-k sampling is retained as a qualitative decoding comparison because it is stochastic.

In [ ]:
# ============================================================
# 21. GENERATE TEST PREDICTIONS
# ============================================================

references = []
greedy_hypotheses = []
beam_hypotheses = []
records = []

for i in tqdm(
    range(len(test_df)),
    desc="Test decoding"
):
    source = test_df.iloc[i]["src"]
    reference = test_df.iloc[i]["trg"]

    src = prepare_source_sentence(source)

    # Full held-out test set, same examples for quantitative comparison.
    greedy_ids, _ = greedy_decode(
        attention_model,
        src,
        max_len=MAX_DECODE_LEN
    )

    beam_ids = beam_search_decode(
        attention_model,
        src,
        beam_width=BEAM_WIDTH,
        max_len=MAX_DECODE_LEN
    )

    greedy_text = ids_to_sentence(greedy_ids)
    beam_text = ids_to_sentence(beam_ids)

    references.append([tokenize(reference)])
    greedy_hypotheses.append(tokenize(greedy_text))
    beam_hypotheses.append(tokenize(beam_text))

    records.append({
        "source": source,
        "reference": reference,
        "greedy_prediction": greedy_text,
        "beam_prediction": beam_text
    })


In [ ]:
# ============================================================
# 22. BLEU
# ============================================================

greedy_bleu = corpus_bleu(
    references,
    greedy_hypotheses
)

beam_bleu = corpus_bleu(
    references,
    beam_hypotheses
)

print(
    f"Bahdanau Attention + Greedy BLEU: "
    f"{greedy_bleu * 100:.2f}"
)

print(
    f"Bahdanau Attention + Beam-5 BLEU: "
    f"{beam_bleu * 100:.2f}"
)

In [ ]:
# ============================================================
# 23. RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(records)

results_df.to_csv(
    "experiment2_attention_test_predictions.csv",
    index=False
)

decoding_comparison = pd.DataFrame({
    "Model": [
        "Bahdanau Attention",
        "Bahdanau Attention"
    ],
    "Decoding": [
        "Greedy",
        "Beam Search (5)"
    ],
    "BLEU": [
        greedy_bleu * 100,
        beam_bleu * 100
    ]
})

display(decoding_comparison)

decoding_comparison.to_csv(
    "experiment2_decoding_comparison.csv",
    index=False
)

# 24. Sentence-Level and Length Analysis

Corpus BLEU is supplemented with sentence-level BLEU and source-length analysis.

This helps test the expected hypothesis that attention reduces the fixed-context bottleneck, especially for longer sentences.

In [ ]:
# ============================================================
# 24. SENTENCE BLEU + LENGTH ANALYSIS
# ============================================================

smooth = SmoothingFunction().method1

sentence_scores = []

for i in range(len(results_df)):
    reference = [tokenize(results_df.iloc[i]["reference"])]
    hypothesis = tokenize(results_df.iloc[i]["greedy_prediction"])

    score = sentence_bleu(
        reference,
        hypothesis,
        smoothing_function=smooth
    )

    sentence_scores.append(score)

results_df["sentence_bleu"] = sentence_scores

results_df["source_length"] = results_df["source"].apply(
    lambda x: len(tokenize(x))
)

results_df["reference_length"] = results_df["reference"].apply(
    lambda x: len(tokenize(x))
)

results_df["prediction_length"] = results_df["greedy_prediction"].apply(
    lambda x: len(tokenize(x))
)

results_df["length_ratio"] = (
    results_df["prediction_length"]
    / results_df["reference_length"].replace(0, 1)
)


def length_bucket(n):
    if n <= 5:
        return "1-5"
    if n <= 10:
        return "6-10"
    if n <= 20:
        return "11-20"
    if n <= 30:
        return "21-30"
    return "31+"


results_df["length_bucket"] = (
    results_df["source_length"].apply(length_bucket)
)

length_analysis = (
    results_df
    .groupby("length_bucket")["sentence_bleu"]
    .mean()
    .reset_index()
)

display(length_analysis)

In [ ]:
# ============================================================
# 25. LENGTH PLOT
# ============================================================

plt.figure(figsize=(9, 5))

plt.bar(
    length_analysis["length_bucket"],
    length_analysis["sentence_bleu"]
)

plt.xlabel("Source sentence length")
plt.ylabel("Mean sentence BLEU")
plt.title("Attention Model BLEU by Source Sentence Length")
plt.grid(axis="y", alpha=0.3)
plt.show()

In [ ]:
# ============================================================
# 26. FAILURE ANALYSIS
# ============================================================

# Rare/unseen vocabulary:
results_df["contains_unk"] = (
    results_df["greedy_prediction"]
    .str.contains(re.escape(UNK_TOKEN), regex=True)
)

unk_rate = results_df["contains_unk"].mean() * 100


def has_repetition(sentence):
    tokens = tokenize(sentence)
    return any(
        tokens[i] == tokens[i+1]
        for i in range(len(tokens) - 1)
    )


results_df["repetition"] = (
    results_df["greedy_prediction"]
    .apply(has_repetition)
)

repetition_rate = results_df["repetition"].mean() * 100

print(f"<unk> prediction rate: {unk_rate:.2f}%")
print(f"Adjacent repetition rate: {repetition_rate:.2f}%")
print(
    "Mean prediction/reference length ratio:",
    results_df["length_ratio"].mean()
)

# 27. Qualitative Failure Examples

Inspect both the best and worst translations.

When writing the report, explicitly identify:

- long-sentence failures
- rare-word failures
- omissions
- additions
- repeated words
- incorrect word order
- premature end-of-sentence
- cases where beam search improves or worsens greedy decoding

In [ ]:
# ============================================================
# 27. BEST / WORST EXAMPLES
# ============================================================

best_examples = results_df.sort_values(
    "sentence_bleu",
    ascending=False
).head(10)

worst_examples = results_df.sort_values(
    "sentence_bleu",
    ascending=True
).head(10)

print("=" * 80)
print("BEST EXAMPLES")
print("=" * 80)

for _, row in best_examples.iterrows():
    print("\nSOURCE:", row["source"])
    print("REFERENCE:", row["reference"])
    print("GREEDY:", row["greedy_prediction"])
    print("BEAM:", row["beam_prediction"])
    print("BLEU:", round(row["sentence_bleu"], 4))

print("\n" + "=" * 80)
print("WORST EXAMPLES")
print("=" * 80)

for _, row in worst_examples.iterrows():
    print("\nSOURCE:", row["source"])
    print("REFERENCE:", row["reference"])
    print("GREEDY:", row["greedy_prediction"])
    print("BEAM:", row["beam_prediction"])
    print("BLEU:", round(row["sentence_bleu"], 4))

# 28. Attention Visualization

The attention matrix provides an interpretable view of which source tokens influenced each generated target token.

Rows correspond to generated French tokens; columns correspond to English source tokens.

In [ ]:
# ============================================================
# 28. ATTENTION HEATMAP
# ============================================================

def visualize_attention(sentence):
    src = prepare_source_sentence(sentence)

    generated_ids, history = greedy_decode(
        attention_model,
        src
    )

    source_tokens = tokenize(sentence)
    target_tokens = trg_vocab.decode(generated_ids)

    matrix = np.array(history)
    matrix = matrix[
        :len(target_tokens),
        :len(source_tokens)
    ]

    plt.figure(
        figsize=(
            max(8, 0.6 * len(source_tokens)),
            max(4, 0.5 * len(target_tokens))
        )
    )

    plt.imshow(matrix, aspect="auto")

    plt.xticks(
        range(len(source_tokens)),
        source_tokens,
        rotation=45,
        ha="right"
    )

    plt.yticks(
        range(len(target_tokens)),
        target_tokens
    )

    plt.xlabel("English source tokens")
    plt.ylabel("Generated French tokens")
    plt.title("Bahdanau Attention Weights")
    plt.colorbar(label="Attention weight")
    plt.tight_layout()
    plt.show()


attention_example = test_df.iloc[0]["src"]
print("Source:", attention_example)
visualize_attention(attention_example)

# 29. Task 1 vs Task 2 BLEU

Enter the **actual Task 1 corpus BLEU** below.

The notebook intentionally does not invent a Task 1 result. Once the actual value is entered, the final comparison table is generated automatically.

Example:

```python
EXP1_BLEU = 12.34
```

In [ ]:
# ============================================================
# 29. FINAL BLEU COMPARISON
# ============================================================

# Replace None with the actual Task-1 corpus BLEU.
EXP1_BLEU = None

if EXP1_BLEU is not None:

    comparison = pd.DataFrame({
        "Model": [
            "Task 1 — No Attention",
            "Task 2 — Bahdanau Attention",
            "Task 2 — Bahdanau Attention"
        ],
        "Decoding": [
            "Greedy",
            "Greedy",
            "Beam Search (5)"
        ],
        "BLEU": [
            EXP1_BLEU,
            greedy_bleu * 100,
            beam_bleu * 100
        ]
    })

    display(comparison)

    comparison.to_csv(
        "task1_vs_task2_bleu_comparison.csv",
        index=False
    )

else:
    print(
        "Set EXP1_BLEU to the actual Task-1 BLEU "
        "to generate the final comparison."
    )

# 30. Final Discussion

## Architecture improvement

Task 1 compressed the source sentence into a fixed-size recurrent representation. Task 2 retains all encoder states and computes a dynamic context vector for every decoder step.

Therefore:

\[
\text{fixed context}
\rightarrow
\text{dynamic attention context}
\]

## Decoding improvement

Greedy decoding makes one locally optimal choice at a time. Beam search keeps multiple candidate translations and performs broader sequence-level search.

Therefore:

\[
\text{greedy search}
\rightarrow
\text{beam search}
\]

## Expected findings

Attention should be especially useful for longer sentences because the decoder no longer has to recover all source information from a single fixed vector.

Remaining failures can still arise from:

- rare or unseen words
- vocabulary `<unk>` replacement
- ambiguous language
- word-order differences
- repetition
- premature termination
- exposure bias from teacher forcing

The report should use the actual BLEU numbers and several actual test examples rather than claiming an improvement before running the experiment.

In [ ]:
# ============================================================
# 30. FINAL SUMMARY + SAVE ARTIFACTS
# ============================================================

print("=" * 80)
print("EXPERIMENT 2 — FINAL SUMMARY")
print("=" * 80)

print(f"Attention + Greedy BLEU : {greedy_bleu * 100:.2f}")
print(f"Attention + Beam-5 BLEU : {beam_bleu * 100:.2f}")
print(f"<unk> prediction rate   : {unk_rate:.2f}%")
print(f"Repetition rate         : {repetition_rate:.2f}%")
print(
    f"Mean length ratio      : "
    f"{results_df['length_ratio'].mean():.4f}"
)

# Final model checkpoint.
torch.save(
    attention_model.state_dict(),
    "experiment2_attention_final.pt"
)

# Detailed predictions.
results_df.to_csv(
    "experiment2_attention_results.csv",
    index=False
)

# Decoding comparison.
decoding_comparison.to_csv(
    "experiment2_decoding_strategy_comparison.csv",
    index=False
)

print("\nSaved artifacts:")
print("  experiment2_attention_final.pt")
print("  experiment2_attention_results.csv")
print("  experiment2_decoding_strategy_comparison.csv")
print("  experiment2_attention_test_predictions.csv")

## Runtime Optimization Summary

The final notebook is configured for faster execution while preserving the assignment requirements:

| Component | Updated setting |
|---|---|
| Encoder/decoder embedding | 128 |
| Hidden dimension | 256 |
| Batch size | 64 |
| Maximum epochs | 10 |
| Early stopping | patience 2 |
| Source/target training limit | 60 tokens |
| Maximum decode length | 60 tokens |
| Beam width | 5 |
| Top-k | 5 |
| Top-k temperature | 0.8 |
| Attention | Bahdanau additive attention |
| Attention masking | Yes |
| Greedy decoding | Yes |
| Beam search | Yes |
| Top-k sampling | Yes |
| Test BLEU | Full held-out test set |

The test comparison still uses the same held-out examples for greedy and Beam-5, so their BLEU scores remain directly comparable within this experiment.


# 31. Submission Checklist

- [x] Same dataset as Task 1
- [x] Same train/validation/test split
- [x] Training-only vocabulary
- [x] LSTM encoder
- [x] Bahdanau attention
- [x] Padding mask in attention
- [x] Dynamic context vector
- [x] Attention decoder
- [x] Teacher forcing
- [x] Validation-based checkpoint selection
- [x] Greedy decoding
- [x] Beam search
- [x] Top-k sampling
- [x] Test-set BLEU
- [x] Decoding comparison
- [x] No-attention vs attention comparison
- [x] Sentence-length analysis
- [x] Rare-word analysis
- [x] Repetition analysis
- [x] Example outputs
- [x] Attention heatmap
- [x] Saved model/predictions

**Important:** run the notebook before submission and replace `EXP1_BLEU = None` with the actual Task-1 BLEU. Do not fabricate numerical results.